# Data Quality Check with a FAIR Dataset Using Tableau Prep

## Goal
Choose one publicly available FAIR dataset and use Tableau Prep to explore and improve its data quality based on the topics discussed in this chapter. The chapter focuses on key data quality dimensions such as completeness, consistency, timeliness, uniqueness, integrity, validity, relevance, and traceability, as well as common issues like missing values, duplicates, inconsistent formats, and invalid data.

## Task
Select one public dataset with a clear source and use Tableau Prep to identify and address data quality issues using only the Tableau Prep features introduced in class so far, such as loading data, renaming fields, filtering, changing data types, simple calculated fields, validation, and exporting results.
What you should do:
- Choose one publicly available FAIR dataset.
- Briefly describe the dataset:
    - title
    - source/link
    - what the dataset contains
    - why you selected it
- Identify relevant data quality issues, for example:
    - missing values
    - duplicate records
    - inconsistent formats
    - invalid values
    - incorrect data types
    - irrelevant fields
- Use Tableau Prep to perform a small cleaning workflow based on the methods introduced in class.
- Explain which data quality dimensions are involved in your dataset and how your cleaning steps help improve them.
- Summarise:
    - the main problems found
    - the steps you applied
    - the improvements achieved
    - any remaining limitations

## Important Note
Use only the Tableau Prep functions and techniques covered in class so far.
Keep the workflow simple and aligned with the course material already discussed.
Submission

Please submit the following via Moodle:
   - Tableau Prep flow file
   - cleaned output file (.csv)
   - short report (1–2 pages) including:
        - dataset description
        - identified data quality issues
        - screenshots of the workflow
        - explanation of the cleaning steps
        - short conclusion

In [8]:
import pandas as pd

df = pd.read_csv("../../data/london_underground_metadata.csv")

# Shape and column names
print(f"Rows: {df.shape[0]}  and  Cols: {df.shape[1]}")
print(f"Columns: {list(df.columns)}")
# Inferred data types
print("\nData types:")
df.dtypes

Rows: 272  and  Cols: 18
Columns: ['station_id', 'station_name', 'lines_served', 'local_authority', 'zone', 'opening_year', 'annual_usage_millions_2024', 'area_served', 'note', 'latitude', 'longitude', 'easting_utm_m', 'northing_utm_m', 'line_count', 'is_interchange', 'usage_rank_2024', 'usage_band_2024', 'step_free_dec2025']

Data types:


station_id                        str
station_name                      str
lines_served                      str
local_authority                   str
zone                              str
opening_year                    int64
annual_usage_millions_2024    float64
area_served                       str
note                              str
latitude                      float64
longitude                     float64
easting_utm_m                 float64
northing_utm_m                float64
line_count                      int64
is_interchange                   bool
usage_rank_2024                 int64
usage_band_2024                   str
step_free_dec2025                bool
dtype: object

## 1. Missing values & duplicates

In [9]:
# Null counts — expect all zero given no NaNs found
display(df.isnull().sum().to_frame("nulls")[lambda x: x["nulls"] > 0])

# Duplicate rows and IDs
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate station_id: {df.duplicated('station_id').sum()}")

# Recurring names with distinct IDs — legitimate per README
(df.groupby("station_name")["station_id"]
   .nunique()
   .rename("distinct_ids")
   .loc[lambda x: x > 1]
   .to_frame())

,nulls


Duplicate rows: 0
Duplicate station_id: 0


,distinct_ids
station_name,
Edgware Road,2
Hammersmith,2
Paddington,2


## 2. Invalid values

In [10]:
# Stations with zero usage — closed or anomalous in 2024
# 0.0 is ambiguous: no closure flag to distinguish from missing data
df.loc[df["annual_usage_millions_2024"] == 0,
       ["station_id", "station_name", "annual_usage_millions_2024"]]

,station_id,station_name,annual_usage_millions_2024
53,LU054,Colindale,0.0
123,LU124,Kensington (Olympia),0.0
124,LU125,Kentish Town,0.0


In [11]:
# Opening year range
print(f"Opening year range: {df['opening_year'].min()} – {df['opening_year'].max()}")

# Stations outside Greater London bounding box
# Amersham, Chesham, Chalfont are legitimately in Buckinghamshire
LAT_MIN, LAT_MAX = 51.28, 51.75
LON_MIN, LON_MAX = -0.55, 0.35

df.loc[
    (df["latitude"] < LAT_MIN) | (df["latitude"] > LAT_MAX) |
    (df["longitude"] < LON_MIN) | (df["longitude"] > LON_MAX),
    ["station_id", "station_name", "latitude", "longitude", "zone"]
]

Opening year range: 1863 – 2021


,station_id,station_name,latitude,longitude,zone
4,LU005,Amersham,51.674150,-0.607392,9
41,LU042,Chalfont & Latimer,51.668083,-0.560518,8
45,LU046,Chesham,51.705224,-0.611068,9


## 3. Inconsistent formats

In [12]:
# Multi-zone values use space as separator — no documented standard
df["zone"].value_counts().sort_index().to_frame()

,count
zone,
1,62
1 2,5
2,56
2 3,14
3,35
3 4,6
4,44
5,22
5 6,1


In [18]:
valid_boroughs = [
    "Barking and Dagenham", "Barnet", "Bexley", "Brent", "Bromley",
    "Camden", "City of London", "City of Westminster", "Croydon", "Ealing",
    "Enfield", "Epping Forest", "Greenwich", "Hackney", "Hammersmith and Fulham",
    "Haringey", "Harrow", "Havering", "Hertfordshire", "Hillingdon", "Hounslow",
    "Islington", "Kensington and Chelsea", "Kingston upon Thames", "Lambeth",
    "Lewisham", "Merton", "Newham", "Redbridge", "Richmond upon Thames",
    "Southwark", "Sutton", "Three Rivers", "Tower Hamlets", "Waltham Forest",
    "Wandsworth", "Buckinghamshire"
]

df.loc[
    ~df["local_authority"].isin(valid_boroughs),
    ["station_id", "station_name", "local_authority"]
]

,station_id,station_name,local_authority
4,LU005,Amersham,Bucking­ham­shire
41,LU042,Chalfont & Latimer,Bucking­ham­shire
43,LU044,Chancery Lane,City of London Camden
45,LU046,Chesham,Bucking­ham­shire
126,LU127,Kew Gardens,Richmond
143,LU144,Manor House,Hackney Haringey
169,LU170,Old Street,Islington Hackney
191,LU192,Richmond,Richmond
193,LU194,Roding Valley,Epping Forest Redbridge
224,LU225,Sudbury Town,Brent Ealing


## 4. Hidden characters

In [14]:
# Scan all string columns for soft hyphens and zero-width characters
# U+00AD causes silent string comparison failures
bad_chars = r"[\u00ad\u200b\u200c\u200d\ufeff]"
results = []

for col in df.select_dtypes(include="str").columns:
    hits = df[df[col].str.contains(bad_chars, na=False, regex=True)]
    if not hits.empty:
        results.append(hits[["station_id", "station_name", col]].assign(column=col))

pd.concat(results) if results else print("None found.")

,station_id,station_name,local_authority,column
4,LU005,Amersham,Bucking­ham­shire,local_authority
41,LU042,Chalfont & Latimer,Bucking­ham­shire,local_authority
45,LU046,Chesham,Bucking­ham­shire,local_authority


## 5. Schema & documentation

In [15]:
readme_cols = [
    "station_id", "station_name", "lines_served", "local_authority",
    "zone", "opening_year", "annual_usage_millions_2024", "usage_rank_2024",
    "usage_band_2024", "step_free_dec2025", "step_free_year",
    "latitude", "longitude", "easting_utm_m", "northing_utm_m", "note"
]
csv_cols = [c.lower() for c in df.columns]

print("In README, absent from CSV:", [c for c in readme_cols if c not in csv_cols])
print("In CSV, absent from README: ", [c for c in csv_cols if c not in readme_cols])

In README, absent from CSV: ['step_free_year']
In CSV, absent from README:  ['area_served', 'line_count', 'is_interchange']


In [16]:
# is_interchange is fully derivable from line_count — adds no information
print(f"is_interchange conflicts with line_count > 1: {((df['line_count'] > 1) != df['is_interchange']).sum()}")

# note column — README says non-standardised, check actual uniqueness
print(f"Unique notes: {df['note'].nunique()} out of {len(df)} rows")

is_interchange conflicts with line_count > 1: 0
Unique notes: 74 out of 272 rows


In [17]:
df.drop(columns=["derived_line_count"], inplace=True, errors="ignore")